In [40]:
!pip show great_expectations

In [41]:
# !pip uninstall great_expectations==0.17 -y
# !pip install great_expectations==0.17 -q
!pip install great-expectations==0.18.12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 10.1 MB/s eta 0:00:00m eta 0:00:010:00:01
  Installing build dependdone
  Getting requirements to build wheel ... done
  Installing backend dependencies ... one
^C
anceled
ERROR: Operation cancelled by user


In [34]:
import great_expectations as gx

In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Ex10").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/12 10:44:51 WARN Utils: Your hostname, developer, resolves to a loopback address: 127.0.1.1; using 192.168.29.9 instead (on interface enp2s0)
25/11/12 10:44:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/12 10:44:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df = spark.read.csv("data/202306-divvy-tripdata.csv", header=True, inferSchema=True)

In [8]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [9]:
import great_expectations as gx
from ruamel import yaml
from great_expectations.core.batch import RuntimeBatchRequest

In [17]:
spark.stop()

In [42]:
import pandas as pd

data = {
    "user_id": [1, 2, 3, 4, 5],
    "age": [25, 32, None, 45, 28],
    "email": ["a@x.com", "b@x.com", "c@x.com", None, "e@x.com"],
    "country": ["IN", "US", "US", "UK", "IN"]
}

df = pd.DataFrame(data)

In [46]:
# import great_expectations as gx
# from great_expectations.core.batch import Batch
# from great_expectations.validator.validator import Validator
# from great_expectations.execution_engine import PandasExecutionEngine

# # Initialize Execution Engine
# execution_engine = PandasExecutionEngine()

# # Wrap DataFrame into a Batch
# batch = Batch(data=df)

# # Create a Validator
# validator = Validator(execution_engine=execution_engine, batches=[batch])

# ----------------------------------------------------------

import great_expectations as gx
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine import PandasExecutionEngine

execution_engine = PandasExecutionEngine()

batch = Batch(data=df)

validator = Validator(execution_engine=execution_engine, batches=[batch])

In [47]:
results = validator.validate()
print(results)

Calculating Metrics: 0it [00:00, ?it/s]

{
  "success": true,
  "results": [],
  "suite_name": "default",
  "suite_parameters": {},
  "statistics": {
    "evaluated_expectations": 0,
    "successful_expectations": 0,
    "unsuccessful_expectations": 0,
    "success_percent": null
  },
  "meta": {
    "great_expectations_version": "1.9.0",
    "expectation_suite_name": "default",
    "run_id": {
      "run_name": null,
      "run_time": "2025-11-12T11:31:12.063100+05:30"
    },
    "batch_spec": {},
    "batch_markers": {
      "ge_load_time": "20251112T060059.387536Z"
    },
    "active_batch_definition": {},
    "validation_time": "20251112T060112.062976Z",
    "checkpoint_name": null
  },
  "id": null
}


In [ ]:
# ========================================

In [51]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("BikeTripsQualityCheck") \
    .getOrCreate()

In [52]:
spark_df = spark.read.csv("data/202306-divvy-tripdata.csv", header=True,  inferSchema=True)

In [53]:
from pyspark.sql import functions as F

In [59]:
import great_expectations as gx
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch

execution_engine = SparkDFExecutionEngine(spark_df)


batch = Batch(data=spark_df)

# Create a Validator
validator = Validator(execution_engine=execution_engine, batches=[batch])

In [67]:
validator.expect_column_values_to_not_be_null("ride_id")

# Trips typically within a day (max 26 hours for buffer)
validator.expect_column_values_to_be_between(
    "start_lat", min_value=40, max_value=45
)

# validator.expect_column_pair_values_to_be_in_order(
#     column_A="start_time",
#     column_B="end_time",
#     strict=True
# )

/home/developer/anaconda3/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1599: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "column": "start_lat",
      "min_value": 40.0,
      "max_value": 45.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 999,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [69]:
results = validator.validate()
print(results)

Calculating Metrics:   0%|          | 0/19 [00:00<?, ?it/s]

{
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "column": "start_station_id"
        },
        "meta": {},
        "severity": "critical"
      },
      "result": {
        "element_count": 999,
        "unexpected_count": 977,
        "unexpected_percent": 97.7977977977978,
        "partial_unexpected_list": [
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null
        ]
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_c

In [71]:
import json

def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect trip_duration_hours column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)


🚦 DATA QUALITY VALIDATION SUMMARY 🚦
{
  "evaluated_expectations": 3,
  "successful_expectations": 2,
  "unsuccessful_expectations": 1,
  "success_percent": 66.66666666666666
}

❌ ALERT: Data Quality Checks Failed!
 - Expectation `Unknown expectation` failed for column: start_station_id

⚠️ Recommendation: Inspect trip_duration_hours column for abnormal values.



In [64]:
pd.read_csv("data/202306-divvy-tripdata.csv").info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ride_id             999 non-null    object 
 1   rideable_type       999 non-null    object 
 2   started_at          999 non-null    object 
 3   ended_at            999 non-null    object 
 4   start_station_name  22 non-null     object 
 5   start_station_id    22 non-null     object 
 6   end_station_name    22 non-null     object 
 7   end_station_id      22 non-null     object 
 8   start_lat           999 non-null    float64
 9   start_lng           999 non-null    float64
 10  end_lat             999 non-null    float64
 11  end_lng             999 non-null    float64
 12  member_casual       999 non-null    object 
dtypes: float64(4), object(9)
memory usage: 101.6+ KB


In [72]:
spark.stop()

In [73]:
# ===========================

In [155]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Bike_Max_Trip").getOrCreate()

In [156]:
df = spark.read.csv("data/202306-divvy-tripdata.csv", header=True, inferSchema=True)

In [157]:
import great_expectations as gx
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator

execution_engine = SparkDFExecutionEngine(df)

batch = Batch(data=df)

validator = Validator(execution_engine=execution_engine, batches=[batch])

In [158]:
df = df.withColumn("duration", F.date_diff("ended_at", "started_at"))
df.select("started_at", "ended_at", "duration").show()

+-------------------+-------------------+--------+
|         started_at|           ended_at|duration|
+-------------------+-------------------+--------+
|2023-06-05 13:34:12|2023-06-05 14:31:56|       0|
|2023-06-05 01:30:22|2023-06-05 01:33:06|       0|
|2023-06-20 18:15:49|2023-06-20 18:32:05|       0|
|2023-06-19 14:56:00|2023-06-19 15:00:35|       0|
|2023-06-19 15:03:34|2023-06-19 15:07:16|       0|
|2023-06-09 21:30:25|2023-06-09 21:49:52|       0|
|2023-06-03 13:34:09|2023-06-03 13:34:28|       0|
|2023-06-03 13:34:46|2023-06-03 13:35:00|       0|
|2023-06-02 22:27:35|2023-06-02 22:35:26|       0|
|2023-06-02 21:18:31|2023-06-03 01:27:19|       1|
|2023-06-23 17:26:02|2023-06-23 17:31:08|       0|
|2023-06-23 15:37:49|2023-06-23 16:02:04|       0|
|2023-06-30 18:56:13|2023-06-30 19:30:40|       0|
|2023-06-02 16:24:02|2023-06-02 16:38:43|       0|
|2023-06-02 12:03:13|2023-06-02 12:11:01|       0|
|2023-06-30 06:12:31|2023-06-30 06:23:05|       0|
|2023-06-30 08:28:51|2023-06-30

In [163]:
df.select("duration").groupby("duration").agg(F.count("duration")).show()

+--------+---------------+
|duration|count(duration)|
+--------+---------------+
|       1|              5|
|   27759|              2|
|       0|            992|
+--------+---------------+



In [167]:
df.filter(F.col("duration") == 27759).head()

Row(ride_id='BE9355D987D363F8', rideable_type='electric_bike', started_at=datetime.datetime(2023, 6, 8, 18, 36, 20), ended_at=datetime.datetime(2099, 6, 8, 18, 41, 23), start_station_name=None, start_station_id=None, end_station_name=None, end_station_id=None, start_lat=41.91, start_lng=-87.65, end_lat=41.91, end_lng=-87.63, member_casual='member', duration=27759)

In [80]:
df1 = df

In [82]:
df1.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [152]:
df1 = df1.withColumn("duration", F.date_diff("ended_at", "started_at"))
df1.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- duration: integer (nullable = true)



In [154]:
spark.stop()

In [ ]:
# df1.select("started_at","ended_at","duration").show(truncate=False)
df1.select("started_at","ended_at","duration").show()

In [126]:
df1.select("duration").groupBy("duration").agg(F.count("duration")).show()

+--------+---------------+
|duration|count(duration)|
+--------+---------------+
|      -1|              5|
|  -27759|              2|
|       0|            992|
+--------+---------------+



In [131]:
validator.expect_column_values_to_not_be_in_set("duration", [0])

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

{
  "success": false,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_in_set",
    "kwargs": {
      "column": "duration",
      "value_set": [
        0
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 999,
    "unexpected_count": 992,
    "unexpected_percent": 99.2992992992993,
    "partial_unexpected_list": [
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0
    ],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 99.2992992992993,
    "unexpected_percent_nonmissing": 99.2992992992993
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [132]:
results = validator.validate()

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

In [134]:
import json

validator.expect_column_values_to_not_be_in_set("duration", [0])

def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect trip_duration_hours column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)


🚦 DATA QUALITY VALIDATION SUMMARY 🚦
{
  "evaluated_expectations": 1,
  "successful_expectations": 0,
  "unsuccessful_expectations": 1,
  "success_percent": 0.0
}

❌ ALERT: Data Quality Checks Failed!
 - Expectation `Unknown expectation` failed for column: duration

 Recommendation: Inspect trip_duration_hours column for abnormal values.



In [179]:
spark.stop()
# df.printSchema()

In [180]:
import json
import great_expectations as gx
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Bike_Max_Trip").getOrCreate()

df = spark.read.csv("data/202306-divvy-tripdata.csv", header=True, inferSchema=True)

# df = df.withColumn("duration", F.date_diff("ended_at", "started_at"))
df = df.withColumn("duration",F.unix_timestamp(F.col("ended_at")) - F.unix_timestamp(F.col("started_at"))/360)

# ----
execution_engine = SparkDFExecutionEngine(spark)

batch = Batch(data=df)

validator = Validator(execution_engine=execution_engine, batches=[batch])

validator.expect_column_values_to_not_be_in_set("duration", [0])

results = validator.validate()
# ----
def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect duration column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)

/home/developer/anaconda3/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1599: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.


Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]


🚦 DATA QUALITY VALIDATION SUMMARY 🚦
{
  "evaluated_expectations": 1,
  "successful_expectations": 1,
  "unsuccessful_expectations": 0,
  "success_percent": 100.0
}

 All data quality checks passed successfully.



In [182]:
df.select("duration").groupby("duration").agg(F.count("duration")).count()

999

In [141]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- duration: integer (nullable = true)



In [142]:
df2 = df.withColumn("duration1", 
                   (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 3600)

In [143]:
df2.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- duration1: double (nullable = true)



In [147]:
df2.select("duration1").groupby("duration1").agg(F.count("duration1")).show()

+-------------------+----------------+
|          duration1|count(duration1)|
+-------------------+----------------+
| 0.5908333333333333|               2|
|0.11527777777777778|               1|
|0.06777777777777778|               1|
| 0.2677777777777778|               1|
| 0.3011111111111111|               1|
| 0.9622222222222222|               1|
|0.17472222222222222|               2|
|0.03388888888888889|               1|
|0.24416666666666667|               1|
|0.21638888888888888|               1|
|0.36083333333333334|               1|
|               0.07|               3|
|0.37194444444444447|               1|
|0.11694444444444445|               2|
|0.38416666666666666|               1|
|0.21055555555555555|               1|
|0.21305555555555555|               2|
| 0.5091666666666667|               2|
|0.23194444444444445|               2|
|0.08333333333333333|               1|
+-------------------+----------------+
only showing top 20 rows


In [183]:
spark.stop()

In [184]:
# ==========================
import json
import great_expectations as gx
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Bike_Max_Trip").getOrCreate()

df = spark.read.csv("data/202306-divvy-tripdata.csv", header=True, inferSchema=True)

# df = df.withColumn("duration", F.date_diff("ended_at", "started_at"))
df = df.withColumn("duration",F.unix_timestamp(F.col("ended_at")) - F.unix_timestamp(F.col("started_at"))/360)

# ----
execution_engine = SparkDFExecutionEngine(spark)

batch = Batch(data=df)

validator = Validator(execution_engine=execution_engine, batches=[batch])

validator.expect_column_values_to_not_be_in_set("duration", [0])

results = validator.validate()
# ----
def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect duration column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)

In [232]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_timestamp,
    unix_timestamp,
    sum as _sum,  # Imported as _sum
    date_format,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)
# ---
import json
import great_expectations as gx
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ----
# Create a SparkSession
spark = SparkSession.builder.appName("BikeRideDuration").getOrCreate()

# Define the schema based on the provided CSV structure
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", StringType(), True),
    StructField("ended_at", StringType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

input_csv_path = "data/202306-divvy-tripdata.csv"

df = spark.read.csv(
    input_csv_path,
    header=True,
    schema=schema,
    mode="DROPMALFORMED"
)

df = df.withColumn(
    "started_at", to_timestamp(col("started_at"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "ended_at", to_timestamp(col("ended_at"), "yyyy-MM-dd HH:mm:ss")
)

df = df.withColumn(
    "duration_seconds",
    unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))
)

df = df.withColumn(
    "date", date_format(col("started_at"), "yyyy-MM-dd")
)

# Use _sum instead of sum here to match the import
daily_durations = df.groupBy("date").agg(
    _sum("duration_seconds").alias("total_duration_seconds")
)

daily_durations = daily_durations.withColumn("same_day", F.when(col("total_duration_seconds")/3600>24 ,0).otherwise(1))


execution_engine = SparkDFExecutionEngine(spark)

batch = Batch(data=daily_durations)

validator = Validator(execution_engine=execution_engine, batches=[batch])

validator.expect_column_values_to_not_be_in_set("same_day", [1])

results = validator.validate()
# ----
def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect total_duration_seconds column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)
output_parquet_path = "results/output_file.parquet"


output_parquet_path = "results/output_file.parquet"
daily_durations.coalesce(1).write.mode("overwrite").parquet(output_parquet_path)
daily_durations.coalesce(1).write.csv("results/output_csv", header=True, mode="overwrite")

25/11/12 14:54:36 WARN CacheManager: Asked to cache already cached data.
/home/developer/anaconda3/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1599: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.


Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]


🚦 DATA QUALITY VALIDATION SUMMARY 🚦
{
  "evaluated_expectations": 1,
  "successful_expectations": 0,
  "unsuccessful_expectations": 1,
  "success_percent": 0.0
}

❌ ALERT: Data Quality Checks Failed!
 - Expectation `Unknown expectation` failed for column: same_day

 Recommendation: Inspect total_duration_seconds column for abnormal values.



In [228]:
daily_durations.coalesce

+----------+----------------------+--------+
|      date|total_duration_seconds|same_day|
+----------+----------------------+--------+
|2023-06-07|                 40262|       1|
|2023-06-02|                 43048|       1|
|2023-06-24|                 25553|       1|
|2023-06-27|                 13848|       1|
|2023-06-08|            2398389723|       0|
|2023-06-29|                 14919|       1|
|2023-06-20|                 23482|       1|
|2023-06-25|            2398403161|       0|
|2023-06-05|                 27141|       1|
|2023-06-04|                 21962|       1|
|2023-06-16|                 36529|       1|
|2023-06-10|                 56995|       1|
|2023-06-11|                  9035|       1|
|2023-06-01|                 45345|       1|
|2023-06-19|                 31995|       1|
|2023-06-13|                 17048|       1|
|2023-06-14|                  8810|       1|
|2023-06-18|                 26773|       1|
|2023-06-30|                 40948|       1|
|2023-06-1

In [226]:
spark.stop()

In [198]:
daily_durations = daily_durations.withColumn("date", F.to_date(col("date")))

In [204]:
from pyspark.sql import *

In [220]:
daily1 = daily_durations.withColumn("same_day", F.when(col("total_duration_seconds")/3600>24 ,0).otherwise(1))

In [225]:
daily1.show()

+----------+----------------------+--------+
|      date|total_duration_seconds|same_day|
+----------+----------------------+--------+
|2023-06-07|                 40262|       1|
|2023-06-02|                 43048|       1|
|2023-06-24|                 25553|       1|
|2023-06-27|                 13848|       1|
|2023-06-08|            2398389723|       0|
|2023-06-29|                 14919|       1|
|2023-06-20|                 23482|       1|
|2023-06-25|            2398403161|       0|
|2023-06-05|                 27141|       1|
|2023-06-04|                 21962|       1|
|2023-06-16|                 36529|       1|
|2023-06-10|                 56995|       1|
|2023-06-11|                  9035|       1|
|2023-06-01|                 45345|       1|
|2023-06-19|                 31995|       1|
|2023-06-13|                 17048|       1|
|2023-06-14|                  8810|       1|
|2023-06-18|                 26773|       1|
|2023-06-30|                 40948|       1|
|2023-06-1

In [221]:
daily1.select("date", "total_duration_seconds", "same_day").groupby("same_day").agg(F.count("same_day")).show()

+--------+---------------+
|same_day|count(same_day)|
+--------+---------------+
|       1|             28|
|       0|              2|
+--------+---------------+



In [210]:
daily1.filter(F.col("same_day") == 0).head()

Row(date=datetime.date(2023, 6, 7), total_duration_seconds=40262, same_day=0)

In [211]:
daily_durations.show()

+----------+----------------------+
|      date|total_duration_seconds|
+----------+----------------------+
|2023-06-07|                 40262|
|2023-06-02|                 43048|
|2023-06-24|                 25553|
|2023-06-27|                 13848|
|2023-06-08|            2398389723|
|2023-06-29|                 14919|
|2023-06-20|                 23482|
|2023-06-25|            2398403161|
|2023-06-05|                 27141|
|2023-06-04|                 21962|
|2023-06-16|                 36529|
|2023-06-10|                 56995|
|2023-06-11|                  9035|
|2023-06-01|                 45345|
|2023-06-19|                 31995|
|2023-06-13|                 17048|
|2023-06-14|                  8810|
|2023-06-18|                 26773|
|2023-06-30|                 40948|
|2023-06-15|                 13230|
+----------+----------------------+
only showing top 20 rows


In [214]:
40262/(60*60)

11.18388888888889

In [218]:
daily1.select("same_day").show()

+--------+
|same_day|
+--------+
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
+--------+
only showing top 20 rows


In [219]:
60*60

3600

In [233]:
spark.stop()

In [236]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, sum as _sum, date_format, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json
import great_expectations as gx
from great_expectations.execution_engine import SparkDFExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator

# -------------------------------------
# 1. Spark setup
# -------------------------------------
spark = SparkSession.builder.appName("BikeRideDuration").getOrCreate()

# -------------------------------------
# 2. Schema + data load
# -------------------------------------
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", StringType(), True),
    StructField("ended_at", StringType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

input_csv_path = "data/202306-divvy-tripdata.csv"
df = spark.read.csv(input_csv_path, header=True, schema=schema, mode="DROPMALFORMED")

# -------------------------------------
# 3. Transformations
# -------------------------------------
df = df.withColumn("started_at", to_timestamp(col("started_at"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("ended_at", to_timestamp(col("ended_at"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("duration_seconds", unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) \
       .withColumn("date", date_format(col("started_at"), "yyyy-MM-dd"))

daily_durations = df.groupBy("date").agg(
    _sum("duration_seconds").alias("total_duration_seconds")
)

# same_day = 1 → valid (≤ 24 hr), 0 → abnormal (> 24 hr)
daily_durations = daily_durations.withColumn(
    "same_day",
    when(col("total_duration_seconds") / 3600 > 24, 0).otherwise(1)
)

# -------------------------------------
# 4. Great Expectations setup
# -------------------------------------
context = gx.get_context(mode="ephemeral")                # ✅ add context
execution_engine = SparkDFExecutionEngine(spark)  # ✅ fix arg name
batch = Batch(data=daily_durations)
validator = Validator(execution_engine=execution_engine, batches=[batch])

# -------------------------------------
# 5. Expectations
# -------------------------------------
# Fail if any trip is not same-day (i.e., same_day == 0)
validator.expect_column_values_to_not_be_in_set("same_day", [0])

# -------------------------------------
# 6. Validate
# -------------------------------------
results = validator.validate()

# -------------------------------------
# 7. Console alert
# -------------------------------------
def show_data_quality_alert(results):
    failed = [r for r in results["results"] if not r["success"]]
    
    print("\n🚦 DATA QUALITY VALIDATION SUMMARY 🚦")
    print(json.dumps(results["statistics"], indent=2))
    
    if failed:
        print("\n❌ ALERT: Data Quality Checks Failed!")
        for f in failed:
            
            if isinstance(f["expectation_config"], dict):
                
                exp = f["expectation_config"].get("expectation_type", "Unknown expectation")
            else:
               
                exp = getattr(f["expectation_config"], "expectation_type", "Unknown expectation")
                
           
            if isinstance(f["expectation_config"], dict) and "kwargs" in f["expectation_config"]:
                col = f["expectation_config"]["kwargs"].get("column", "Unknown column")
            else:
                
                kwargs = getattr(f["expectation_config"], "kwargs", {})
                col = kwargs.get("column", "Unknown column") if isinstance(kwargs, dict) else "Unknown column"
                
            print(f" - Expectation `{exp}` failed for column: {col}")
        print("\n Recommendation: Inspect total_duration_seconds column for abnormal values.\n")
    else:
        print("\n All data quality checks passed successfully.\n")

show_data_quality_alert(results)

# -------------------------------------
# 8. Output
# -------------------------------------
output_parquet_path = "results/output_file.parquet"
daily_durations.coalesce(1).write.mode("overwrite").parquet(output_parquet_path)
daily_durations.coalesce(1).write.csv("results/output_csv", header=True, mode="overwrite")


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpckl2un30' for ephemeral docs site
25/11/12 15:02:26 WARN CacheManager: Asked to cache already cached data.


Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]


🚦 DATA QUALITY VALIDATION SUMMARY 🚦
{
  "evaluated_expectations": 1,
  "successful_expectations": 0,
  "unsuccessful_expectations": 1,
  "success_percent": 0.0
}

❌ ALERT: Data Quality Checks Failed!
 - Expectation `Unknown expectation` failed for column: same_day

 Recommendation: Inspect total_duration_seconds column for abnormal values.



In [246]:
# !pip show pyspark

spark.stop()

In [247]:
import great_expectations as gx

In [249]:
!pip show great-expectations

In [1]:
import pandas as pd

In [2]:
import pandas as pd
df = pd.read_parquet('results/output_file.parquet/', engine='pyarrow')

In [11]:
df.head()

,date,total_duration_seconds,same_day
0,2023-06-07,40262,1
1,2023-06-02,43048,1
2,2023-06-24,25553,1
3,2023-06-27,13848,1
4,2023-06-08,2398389723,0


In [8]:
!pip install fastparquet -q

In [9]:
df1 = pd.read_parquet('results/output_file.parquet/', engine='fastparquet')

In [19]:
df1[df1['same_day']==0]

,date,total_duration_seconds,same_day
4,2023-06-08,2398389723,0
7,2023-06-25,2398403161,0


In [15]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   date                    30 non-null     object
 1   total_duration_seconds  30 non-null     int64 
 2   same_day                30 non-null     int32 
dtypes: int32(1), int64(1), object(1)
memory usage: 732.0+ bytes


In [20]:
2398389723/3600

666219.3675

In [21]:
666219.3675/24

27759.140312500003

In [22]:
27759.140312500003/365

76.05243921232878